# DSA 2050 Week 2 Practical Lab

## Part A — Creating the Source Data

In [1]:
import pandas as pd
import sqlite3
import json

In [2]:
#Creating customer data
customers = pd.DataFrame({
    "CustomerID": [
        "C001", "C002", "C003", "C004",
        "C004", "C005", "C006", "C007",
        "C008", "C009", "C010"
    ],

    "CustomerName": [
        "Amina", "Brian", "Carol", "David",
        "David", "Esther", "Faith", "George",
        "Halima", "Ian", "Jane"
    ],

    "Segment": [
        "Retail", "Corporate", "Retail", "SME",
        "SME", "Corporate", "SME", "Retail",
        "Corporate", "Retail", "SME"
    ],

    "RegionCode": [
        "NRB", "MSA", "NRB", "KSM",
        "KSM", "NKR", "NRB", "MSA",
        "KSM", "NKR", "NRB"
    ]
})

customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
#saving customer data as csv
customers.to_csv("customers.csv", index=False)

In [4]:
#Creating orders data
orders = pd.DataFrame({
    "OrderID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015"
    ],

    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],

    "OrderAmount": [
        4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],

    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Office",
        "Electronics", "Home", "Office", "Home", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
    ]
})

orders


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [5]:
#Saving ordes data as csv
orders.to_csv("orders.csv", index=False)

In [6]:
#Creating Json region data
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]

with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)


## Part B — Inspecting the Sources

In [7]:
#Reading the data from csv files
customers = pd.read_csv("customers.csv") 
orders = pd.read_csv("orders.csv")  

In [8]:
#Viewing first 5 rows of the customers data
customers.head()

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [9]:
#Viewing first 5 rows of the orders data
orders.head()

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


In [10]:
#Getting the shape of the customers data
customers.shape

(11, 4)

In [11]:
#Getting the shape of the orders data
orders.shape

(15, 4)

### Q1. What does one row represent in customers.csv?

One row represents one customer record.

### Q2. What does one row represent in orders.csv?

One row represents one order.

### Q3. Are the two datasets at the same aggregation grain? Explain briefly.

No. The two datasets have different aggregation grains. The customers.csv dataset has one row per customer, while the orders.csv dataset has one row per order.


## Part C  - Detecting Key Problems 

In [16]:
#Checking for duplicate CustomerID in customers data
customers["CustomerID"].duplicated().sum()


np.int64(1)

In [17]:
customers["CustomerID"].duplicated(keep=False)

0     False
1     False
2     False
3      True
4      True
5     False
6     False
7     False
8     False
9     False
10    False
Name: CustomerID, dtype: bool

### Q4. Why can a duplicate CustomerID be dangerous when joining customer and order tables?

 A duplicate CustomerID can cause rows to be duplicated during a JOIN. This can increase the number of rows and lead to incorrect totals and misleading analytical results.

### Q5. Which CustomerID appears in orders but not in customers?

 C999

### Q6. Why should an unmatched record be investigated rather than silently deleted?

An unmatched record should be investigated because it may indicate a data quality or integration problem. Silently deleting it could remove valid information and affect the accuracy of the analysis.


## Part D - CREATing the SQLite Database

In [18]:
#Creating a connection to the SQLite database
conn = sqlite3.connect("retail_lab.db")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)


15

In [19]:
#Testing the connection and reading data from the database
query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


## Part E - Select and Where queries

In [20]:
#Selecting specific columns from the orders table
query = """
SELECT
    OrderID,
    CustomerID,
    OrderAmount
FROM orders;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


In [21]:
# Filtering data from the customers table retail segment
query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [ ]:
# Filtering data from orders table where order amount is greater than 8000
query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [24]:
# Filtering data from orders table where product category is Electronics
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [25]:
# Filtering data from orders table where product category is Electronics and order amount is greater than 8000
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
AND OrderAmount > 8000;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


## Part F — GROUP BY and Aggregate Functions



In [26]:
# Total sales 
query = """
SELECT
    SUM(OrderAmount) AS TotalSales
FROM orders;
"""

pd.read_sql_query(query, conn)

,TotalSales
0,113500


In [29]:
# Total sales by product category
query = """
SELECT
   ProductCategory,
    SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


In [27]:
# KPI's by product category
query = """
SELECT
    ProductCategory,
    COUNT(OrderID) AS NumberOfOrders,
    SUM(OrderAmount) AS TotalSales,
    AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""

pd.read_sql_query(query, conn)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


### Q7. Write two short observations about category performance.


1. Electronics recorded the highest total sales at KSh 51,500 from 6 orders, with an average order value of approximately KSh 8,583.33.

2. Home recorded the lowest total sales at KSh 19,500 from 4 orders, while Office recorded KSh 42,500 from 5 orders.

## Part G - JOIN Customers and Orders 

In [30]:
# Connecting orders to customer info
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
    ON o.CustomerID = c.CustomerID;
"""

joined = pd.read_sql_query(query, conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [31]:
# checking the number of rows in orders and joined dataframes
len(orders),len(joined)

(15, 16)

**Q8. Are the row counts equal? If not, explain exactly what caused the difference.**

No. The orders table has 15 rows while the joined table has 16 rows because duplicate `CustomerID C004` caused `O005` to appear twice.

**Q9. Locate the unmatched customer record in the joined output. What values appear for CustomerName and Segment?**

 The unmatched record is `C999` (`O015`). `CustomerName` and `Segment` both appear as None.


## Part H - Reconciling Totals

In [32]:
orders["OrderAmount"].sum()

np.int64(113500)

In [33]:
joined["OrderAmount"].sum()

np.int64(122600)

### Q10. Are the two sales totals equal?

No. Before the JOIN, total sales were 113,500. After the JOIN, total sales were 122,600.

### Q11. If the total changed, which customer caused the duplication?

Customer C004 caused the duplication because order O005 was counted twice.

### Q12. Why is total reconciliation an important validation step after a JOIN?

It helps detect duplicate records that can incorrectly increase the results.

## Part I - Correcting the Duplicate key


In [34]:
# Cleaning the customers data by removing duplicate CustomerID entries
customers_clean = (
    customers
    .drop_duplicates(subset="CustomerID")
)

customers_clean.to_sql(
    "customers_clean",
    conn,
    if_exists="replace",
    index=False
)

10

In [35]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""

clean_joined = pd.read_sql_query(query, conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [38]:
len(orders), len(clean_joined)

(15, 15)

In [37]:
orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum()

(np.int64(113500), np.int64(113500))

### Q13. Has the corrected JOIN preserved both the row count and the total sales value?
Yes. The corrected JOIN preserved the row count at 15 and the total sales value at 113,500. Removing the duplicate C004 record prevented O005 from being counted twice.

## Part J - Customer Segment KPI's


In [39]:
#Calculating KPI's per segment
query = """
SELECT
    c.Segment,
    COUNT(o.OrderID) AS NumberOfOrders,
    SUM(o.OrderAmount) AS TotalSales,
    AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID
GROUP BY c.Segment;
"""

segment_kpis = pd.read_sql_query(query, conn)
segment_kpis

,Segment,NumberOfOrders,TotalSales,AverageOrderValue
0,None,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


### Q14. Write three sentences explaining what management can learn from the segment KPIs.

1.Corporate generated the highest total sales at 46,800 from 4 orders. 

2.Retail had the highest number of orders with 6, while Corporate had the highest average order value at 11,700. 

3.The unmatched customer contributed 9,900 in sales but has no identified segment.

## Part K - Merging the JSON Region Lookup

In [41]:
#Loading regions data from JSON file
regions_df = pd.read_json("regions.json")
regions_df

,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [42]:
# Merging customers_clean with regions_df to get region names
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left"
)

customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


### Q15. Confirm that each customer now has a readable RegionName rather than only a RegionCode.

Each customer has a readable RegionName.

### Q16. If a RegionCode did not exist in regions.json, what would a left merge produce for RegionName?

The RegionName would appear as NULL/None because no matching region would be found.

## Regional Sales Analysis

In [43]:
regional_sales = (
    clean_joined
    .merge(
        customers_regions[["CustomerID", "RegionName"]],
        on="CustomerID",
        how="left"
    )
    .groupby("RegionName", dropna=False)
    .agg(
        NumberOfOrders=("OrderID", "count"),
        TotalSales=("OrderAmount", "sum")
    )
    .reset_index()
)

regional_sales

,RegionName,NumberOfOrders,TotalSales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400
4,NaN,1,9900


### Q17. Which region generated the highest sales?

Nairobi generated the highest sales, with total sales of 39,800.

### Q18. Should management automatically conclude that this is the “best-performing” region? Explain using at least two limitations of the available data.

**Answer:** No. A region having the highest sales does not necessarily mean it is the best-performing because the analysis does not consider profitability or costs. It also does not account for factors such as market size or customer loyalty and purchasing behavior

## Conclusion

1.The biggest data-quality problem was the duplicate CustomerID C004, which caused order O005 to appear twice after the JOIN.

2.This increased the row count from 15 to 16 and the total sales from 113,500 to 122,600. 
 
3.Checking totals after a JOIN is important because it helps identify duplicate records and ensures the results remain accurate. 
 
 4.The Corporate segment generated the highest sales, with total sales of 46,800. 
 
 5.One limitation is that sales alone do not show profitability, costs, or customer loyalty and purchasing behavior.